<a href="https://colab.research.google.com/github/DeepanshuSharma1607/ipl-winner-pedictor/blob/main/ipl.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1908]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [1909]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "IPL_ball_by_ball_updated.csv"

# Load the latest version
df_nn = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "dgsports/ipl-ball-by-ball-2008-to-2022",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

# print("First 5 records:", df__.head())

/tmp/ipykernel_1706/2934442778.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_nn = kagglehub.load_dataset(


Using Colab cache for faster access to the 'ipl-ball-by-ball-2008-to-2022' dataset.


In [1910]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter
import pandas as pd
# Set the path to the file you'd like to load
file_path = "deliveries_updated_ipl_upto_2025.csv"

# Load the latest version
df_mm = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "dgsports/ipl-ball-by-ball-2008-to-2022",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

/tmp/ipykernel_1706/3183163287.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df_mm = kagglehub.load_dataset(


Using Colab cache for faster access to the 'ipl-ball-by-ball-2008-to-2022' dataset.


In [1911]:
import pandas as pd
import numpy as np

# 1. Safely create 'season' without the strict format argument (prevents Pandas version errors)
df_mm['date'] = pd.to_datetime(df_mm['date'], errors='coerce')
df_mm['season'] = df_mm['date'].dt.year

# 2. Filter for 2024 and 2025
df_new = df_mm[df_mm['season'].isin([2024, 2025])].copy()

# 3. Drop existing 'over' and 'ball' columns to avoid duplicates when renaming
if 'over' in df_new.columns:
    df_new = df_new.drop(columns=['over', 'ball'])

# 4. Rename columns to match df_nn
mapping = {
    'matchId': 'match_id',
    'inning': 'innings',
    'over_ball': 'ball',
    'batsman': 'striker',
    'batsman_runs': 'runs_off_bat',
    'dismissal_kind': 'wicket_type',
    'date': 'start_date',
    'isWide': 'wides',
    'isNoBall': 'noballs',
    'Byes': 'byes',
    'LegByes': 'legbyes',
    'Penalty': 'penalty'
}
df_new = df_new.rename(columns=mapping)

# 5. ALIGN COLUMNS SAFELY: This fixes the missing 'venue' error automatically
df_new = df_new.reindex(columns=df_nn.columns)

# 6. Merge the historical df_nn with the new 24/25 data
df_nn_combined = pd.concat([df_nn, df_new], ignore_index=True)

print("Old shape:", df_nn.shape)
print("New combined shape:", df_nn_combined.shape)

Old shape: (243815, 22)
New combined shape: (278203, 22)


In [1912]:
print(df_nn_combined.head(10))

   match_id  season  start_date                  venue  innings  ball  \
0    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   0.1   
1    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   0.2   
2    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   0.3   
3    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   0.4   
4    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   0.5   
5    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   0.6   
6    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   0.7   
7    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   1.1   
8    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   1.2   
9    335982    2008  2008-04-18  M Chinnaswamy Stadium        1   1.3   

            batting_team                 bowling_team      striker  \
0  Kolkata Knight Riders  Royal Challengers Bangalore   SC Ganguly   
1  Kolkata Knight Riders  Royal Challengers Bangalore  B

In [1913]:
# df_matches.shape

In [1914]:
df_nn_combined.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed'],
      dtype='object')

In [1915]:
df_nn_combined.shape

(278203, 22)

In [1916]:
df_nn_combined.isnull().sum()

,0
match_id,0
season,0
start_date,0
venue,34388
innings,0
ball,0
batting_team,0
bowling_team,0
striker,0
non_striker,0


In [1917]:

df=df_nn_combined.copy()

In [1918]:
team_mapping = {
    # Rebranded Teams
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru',

    # Successor Franchises (Optional but recommended for model logic)
    'Deccan Chargers': 'Sunrisers Hyderabad',
    'Rising Pune Supergiants': 'Rising Pune Supergiant',
    'Pune Warriors': 'Rising Pune Supergiant'
}

# Apply to both team columns
df['batting_team'] = df['batting_team'].replace(team_mapping)
df['bowling_team'] = df['bowling_team'].replace(team_mapping)


In [1919]:
df

,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
0,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.1,Kolkata Knight Riders,Royal Challengers Bengaluru,SC Ganguly,BB McCullum,...,1,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.2,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.3,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.4,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278198,1473511,2025,2025-06-03 00:00:00,NaN,2,19.2,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278199,1473511,2025,2025-06-03 00:00:00,NaN,2,19.3,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278200,1473511,2025,2025-06-03 00:00:00,NaN,2,19.4,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278201,1473511,2025,2025-06-03 00:00:00,NaN,2,19.5,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1920]:
df.shape

(278203, 22)

In [1921]:
df

,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,extras,wides,noballs,byes,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed
0,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.1,Kolkata Knight Riders,Royal Challengers Bengaluru,SC Ganguly,BB McCullum,...,1,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN
1,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.2,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.3,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,1,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.4,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278198,1473511,2025,2025-06-03 00:00:00,NaN,2,19.2,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278199,1473511,2025,2025-06-03 00:00:00,NaN,2,19.3,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278200,1473511,2025,2025-06-03 00:00:00,NaN,2,19.4,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
278201,1473511,2025,2025-06-03 00:00:00,NaN,2,19.5,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [1922]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed'],
      dtype='object')

In [1923]:
print(df[['runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed']])

        runs_off_bat  extras  wides  noballs  byes  legbyes  penalty  \
0                  0       1    NaN      NaN   NaN      1.0      NaN   
1                  0       0    NaN      NaN   NaN      NaN      NaN   
2                  0       1    1.0      NaN   NaN      NaN      NaN   
3                  0       0    NaN      NaN   NaN      NaN      NaN   
4                  0       0    NaN      NaN   NaN      NaN      NaN   
...              ...     ...    ...      ...   ...      ...      ...   
278198             0       0    NaN      NaN   NaN      NaN      NaN   
278199             6       0    NaN      NaN   NaN      NaN      NaN   
278200             4       0    NaN      NaN   NaN      NaN      NaN   
278201             6       0    NaN      NaN   NaN      NaN      NaN   
278202             6       0    NaN      NaN   NaN      NaN      NaN   

       wicket_type player_dismissed  other_wicket_type  other_player_dismissed  
0              NaN              NaN                NaN

In [1924]:
cols_to_fix = ['runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes', 'penalty']
df[cols_to_fix] = df[cols_to_fix].fillna(0)

df['curr_ball_runs'] = df['runs_off_bat'] + df['extras']
total_score = df.groupby(['match_id', 'innings', 'batting_team'])['curr_ball_runs'].cumsum().reset_index()


In [1925]:
df['team_runs'] = df.groupby(['match_id','batting_team'])['curr_ball_runs'].cumsum()
df['team_runs'] = df.groupby(['match_id','batting_team'])['team_runs'].shift(1).fillna(0)

In [1926]:
df['first_innings_runs'] = np.where(df['innings'] == 1, df['team_runs'], 0)

df['second_innings_runs'] = np.where(df['innings'] == 2, df['team_runs'], 0)

In [1927]:
df

,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,legbyes,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,curr_ball_runs,team_runs,first_innings_runs,second_innings_runs
0,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.1,Kolkata Knight Riders,Royal Challengers Bengaluru,SC Ganguly,BB McCullum,...,1.0,0.0,NaN,NaN,NaN,NaN,1,0.0,0.0,0.0
1,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.2,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,0.0,NaN,NaN,NaN,NaN,0,1.0,1.0,0.0
2,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.3,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,0.0,NaN,NaN,NaN,NaN,1,1.0,1.0,0.0
3,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.4,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,0.0,NaN,NaN,NaN,NaN,0,2.0,2.0,0.0
4,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,0.0,NaN,NaN,NaN,NaN,0,2.0,2.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278198,1473511,2025,2025-06-03 00:00:00,NaN,2,19.2,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,0.0,NaN,NaN,NaN,NaN,0,162.0,0.0,162.0
278199,1473511,2025,2025-06-03 00:00:00,NaN,2,19.3,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,0.0,NaN,NaN,NaN,NaN,6,162.0,0.0,162.0
278200,1473511,2025,2025-06-03 00:00:00,NaN,2,19.4,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,0.0,NaN,NaN,NaN,NaN,4,168.0,0.0,168.0
278201,1473511,2025,2025-06-03 00:00:00,NaN,2,19.5,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,0.0,NaN,NaN,NaN,NaN,6,172.0,0.0,172.0


In [1928]:
df['is_wicket']=df['player_dismissed'].notna().astype(int)

In [1929]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed', 'curr_ball_runs', 'team_runs',
       'first_innings_runs', 'second_innings_runs', 'is_wicket'],
      dtype='object')

In [1930]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278203 entries, 0 to 278202
Data columns (total 27 columns):
 #   Column                  Non-Null Count   Dtype  
---  ------                  --------------   -----  
 0   match_id                278203 non-null  int64  
 1   season                  278203 non-null  int64  
 2   start_date              278203 non-null  object 
 3   venue                   243815 non-null  object 
 4   innings                 278203 non-null  int64  
 5   ball                    278203 non-null  float64
 6   batting_team            278203 non-null  object 
 7   bowling_team            278203 non-null  object 
 8   striker                 278203 non-null  object 
 9   non_striker             278203 non-null  object 
 10  bowler                  278203 non-null  object 
 11  runs_off_bat            278203 non-null  int64  
 12  extras                  278203 non-null  int64  
 13  wides                   278203 non-null  float64
 14  noballs             

In [1931]:
df

,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,penalty,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,curr_ball_runs,team_runs,first_innings_runs,second_innings_runs,is_wicket
0,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.1,Kolkata Knight Riders,Royal Challengers Bengaluru,SC Ganguly,BB McCullum,...,0.0,NaN,NaN,NaN,NaN,1,0.0,0.0,0.0,0
1,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.2,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,NaN,NaN,NaN,NaN,0,1.0,1.0,0.0,0
2,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.3,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,NaN,NaN,NaN,NaN,1,1.0,1.0,0.0,0
3,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.4,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,NaN,NaN,NaN,NaN,0,2.0,2.0,0.0,0
4,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0.0,NaN,NaN,NaN,NaN,0,2.0,2.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278198,1473511,2025,2025-06-03 00:00:00,NaN,2,19.2,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,NaN,NaN,NaN,NaN,0,162.0,0.0,162.0,0
278199,1473511,2025,2025-06-03 00:00:00,NaN,2,19.3,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,NaN,NaN,NaN,NaN,6,162.0,0.0,162.0,0
278200,1473511,2025,2025-06-03 00:00:00,NaN,2,19.4,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,NaN,NaN,NaN,NaN,4,168.0,0.0,168.0,0
278201,1473511,2025,2025-06-03 00:00:00,NaN,2,19.5,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,0.0,NaN,NaN,NaN,NaN,6,172.0,0.0,172.0,0


In [1932]:
final_scores = df[df['innings'] == 1].groupby('match_id')['team_runs'].max()
df = df.merge(final_scores.rename('target'), on='match_id')
df['target'] = np.where(df['innings'] == 2, df['target'] + 1, 0)

In [1933]:
df['over'] = df['ball'].astype(str).str.split('.').str[0].astype(int)
df['ball_in_over'] = df['ball'].astype(str).str.split('.').str[1].astype(int)

In [1934]:
df['ball_in_over'] = df['ball_in_over'].clip(upper=6)
df['balls_bowled'] = df['over'] * 6 + df['ball_in_over']

In [1935]:
df['runs_to_win'] = np.where(
    df['innings'] == 2,
    df['target'] - df['team_runs'],
    0
)

In [1936]:
df['runs_to_win'] = np.where(
    df['innings'] == 2,
    df['target'] - df['team_runs'],
    0
)

In [1937]:
balls_remaining = (120 - df['balls_bowled']).clip(lower=0)


In [1938]:
df['curr_run_rate'] = np.where(
    df['balls_bowled'] == 0, 0,
    (df['team_runs'] * 6) / df['balls_bowled']
)

In [1939]:
df['req_run_rate'] = np.where(
    balls_remaining == 0,
    np.inf,
    (df['runs_to_win'] * 6) / balls_remaining
)
df['req_run_rate'] = df['req_run_rate'].clip(0, 36)


In [1940]:
df['req_run_rate'] = df['req_run_rate'].replace(np.inf, 100)

In [1941]:
df['crr_rrr_ratio'] = df['curr_run_rate'] / df['req_run_rate'].replace(0, np.inf)

df['crr_rrr_ratio'] = df['crr_rrr_ratio'].clip(0, 10)

df.loc[df['innings'] == 1,
       ['target', 'runs_to_win', 'req_run_rate', 'crr_rrr_ratio']] = 0

In [1942]:
df['balls_remaining'] = (20 * 6) - df['balls_bowled']

In [1943]:
df['total_wickets'] = df.groupby(['match_id','innings'])['is_wicket'].cumsum()
df['total_wickets'] = df.groupby(['match_id','innings'])['total_wickets'].shift(1).fillna(0)
df['wickets_remaining'] = 10 - df['total_wickets']

In [1944]:
df

,match_id,season,start_date,venue,innings,ball,batting_team,bowling_team,striker,non_striker,...,over,ball_in_over,balls_bowled,runs_to_win,curr_run_rate,req_run_rate,crr_rrr_ratio,balls_remaining,total_wickets,wickets_remaining
0,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.1,Kolkata Knight Riders,Royal Challengers Bengaluru,SC Ganguly,BB McCullum,...,0,1,1,0.0,0.000000,0.0,0.000000,119,0.0,10.0
1,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.2,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,2,2,0.0,3.000000,0.0,0.000000,118,0.0,10.0
2,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.3,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,3,3,0.0,2.000000,0.0,0.000000,117,0.0,10.0
3,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.4,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,4,4,0.0,3.000000,0.0,0.000000,116,0.0,10.0
4,335982,2008,2008-04-18,M Chinnaswamy Stadium,1,0.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,SC Ganguly,...,0,5,5,0.0,2.400000,0.0,0.000000,115,0.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
278198,1473511,2025,2025-06-03 00:00:00,NaN,2,19.2,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,19,2,116,29.0,8.379310,36.0,0.232759,4,7.0,3.0
278199,1473511,2025,2025-06-03 00:00:00,NaN,2,19.3,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,19,3,117,29.0,8.307692,36.0,0.230769,3,7.0,3.0
278200,1473511,2025,2025-06-03 00:00:00,NaN,2,19.4,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,19,4,118,23.0,8.542373,36.0,0.237288,2,7.0,3.0
278201,1473511,2025,2025-06-03 00:00:00,NaN,2,19.5,Punjab Kings,Royal Challengers Bengaluru,Shashank Singh,KA Jamieson,...,19,5,119,19.0,8.672269,36.0,0.240896,1,7.0,3.0


In [1945]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed', 'curr_ball_runs', 'team_runs',
       'first_innings_runs', 'second_innings_runs', 'is_wicket', 'target',
       'over', 'ball_in_over', 'balls_bowled', 'runs_to_win', 'curr_run_rate',
       'req_run_rate', 'crr_rrr_ratio', 'balls_remaining', 'total_wickets',
       'wickets_remaining'],
      dtype='object')

In [1946]:
df['is_out'] = (df['player_dismissed'] == df['striker']).astype(int)

df['valid_ball'] = (df['wides'] == 0).astype(int)

# df['batsman_run_cum'] = df.groupby(['match_id','striker'])['runs_off_bat'].cumsum()
# df['batsman_run_cum'] = df.groupby(['match_id','striker'])['batsman_run_cum'].shift(1).fillna(0)

# df['balls_faced_cum'] = df.groupby(['match_id','striker'])['valid_ball'].cumsum()
# df['balls_faced_cum'] = df.groupby(['match_id','striker'])['balls_faced_cum'].shift(1).fillna(0)

# df['match_strike_rate'] = np.where(
#     df['balls_faced_cum'] == 0,
#     0,
#     (df['batsman_run_cum'] / df['balls_faced_cum']) * 100
# )



# df['batsman_run_cum_career'] = df.groupby('striker')['runs_off_bat'].cumsum()
# df['batsman_run_cum_career'] = df.groupby('striker')['batsman_run_cum_career'].shift(1).fillna(0)

# df['balls_faced_cum_career'] = df.groupby('striker')['valid_ball'].cumsum()
# df['balls_faced_cum_career'] = df.groupby('striker')['balls_faced_cum_career'].shift(1).fillna(0)

# df['batsman_outs_career'] = df.groupby('striker')['is_out'].cumsum()
# df['batsman_outs_career'] = df.groupby('striker')['batsman_outs_career'].shift(1).fillna(0)

# universal_runs = 50
# universal_outs = 2

# df['batting_average'] = (
#     df['batsman_run_cum_career'] + universal_runs
# ) / (
#     df['batsman_outs_career'] + universal_outs
# )


# universal_sr_runs = 100
# universal_balls = 80

# df['career_strike_rate'] = (
#     (df['batsman_run_cum_career'] + universal_sr_runs) /
#     (df['balls_faced_cum_career'] + universal_balls)
# ) * 100
# Aggregate per striker per season
season_bat = df.groupby(['season', 'striker']).agg(
    season_runs=('runs_off_bat', 'sum'),
    season_balls=('valid_ball', 'sum'),
    season_outs=('is_out', 'sum')
).reset_index().sort_values(['striker', 'season'])

# Cumsum then shift — so current season stats NOT included
season_bat['career_runs'] = season_bat.groupby('striker')['season_runs'].cumsum().shift(1).fillna(0)
season_bat['career_balls'] = season_bat.groupby('striker')['season_balls'].cumsum().shift(1).fillna(0)
season_bat['career_outs'] = season_bat.groupby('striker')['season_outs'].cumsum().shift(1).fillna(0)

# Smoothed stats
season_bat['batting_average'] = (season_bat['career_runs'] + 50) / (season_bat['career_outs'] + 2)
season_bat['career_strike_rate'] = ((season_bat['career_runs'] + 100) / (season_bat['career_balls'] + 80)) * 100

# Merge back
df = df.merge(
    season_bat[['season', 'striker', 'batting_average', 'career_strike_rate']],
    on=['season', 'striker'], how='left'
)

In [1947]:


# df['bowler_runs_conceded'] = df.groupby('bowler')['runs_off_bat'].cumsum()
# df['bowler_runs_conceded'] = df.groupby('bowler')['bowler_runs_conceded'].shift(1).fillna(0)

# df['bowler_balls_bowled'] = df.groupby('bowler')['valid_ball'].cumsum()
# df['bowler_balls_bowled'] = df.groupby('bowler')['bowler_balls_bowled'].shift(1).fillna(0)

# df['bowler_wickets_cum'] = df.groupby('bowler')['is_wicket'].cumsum()
# df['bowler_wickets_cum'] = df.groupby('bowler')['bowler_wickets_cum'].shift(1).fillna(0)



# universal_runs_conceded = 300
# universal_balls = 240
# universal_wickets = 10


# df['exp_bowler_eco'] = (
#     (df['bowler_runs_conceded'] + universal_runs_conceded) /
#     (df['bowler_balls_bowled'] + universal_balls)
# ) * 6


# df['exp_bowler_avg'] = (
#     df['bowler_runs_conceded'] + universal_runs_conceded
# ) / (
#     df['bowler_wickets_cum'] + universal_wickets
# )
# Aggregate per bowler per season
season_bowl = df.groupby(['season', 'bowler']).agg(
    season_runs_c=('runs_off_bat', 'sum'),
    season_balls_b=('valid_ball', 'sum'),
    season_wkts=('is_wicket', 'sum')
).reset_index().sort_values(['bowler', 'season'])

# Cumsum then shift — so current season NOT included
season_bowl['career_runs_c'] = season_bowl.groupby('bowler')['season_runs_c'].cumsum().shift(1).fillna(0)
season_bowl['career_balls_b'] = season_bowl.groupby('bowler')['season_balls_b'].cumsum().shift(1).fillna(0)
season_bowl['career_wkts'] = season_bowl.groupby('bowler')['season_wkts'].cumsum().shift(1).fillna(0)

# Smoothed stats
season_bowl['exp_bowler_eco'] = ((season_bowl['career_runs_c'] + 300) / (season_bowl['career_balls_b'] + 240)) * 6
season_bowl['exp_bowler_avg'] = (season_bowl['career_runs_c'] + 300) / (season_bowl['career_wkts'] + 10)

# Merge back
df = df.merge(
    season_bowl[['season', 'bowler', 'exp_bowler_eco', 'exp_bowler_avg']],
    on=['season', 'bowler'], how='left'
)

In [1948]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed', 'curr_ball_runs', 'team_runs',
       'first_innings_runs', 'second_innings_runs', 'is_wicket', 'target',
       'over', 'ball_in_over', 'balls_bowled', 'runs_to_win', 'curr_run_rate',
       'req_run_rate', 'crr_rrr_ratio', 'balls_remaining', 'total_wickets',
       'wickets_remaining', 'is_out', 'valid_ball', 'batting_average',
       'career_strike_rate', 'exp_bowler_eco', 'exp_bowler_avg'],
      dtype='object')

In [1949]:
df.shape

(278203, 44)

In [1950]:
final_scores = df.groupby(['match_id', 'innings'])['team_runs'].max().reset_index()
final_scores = final_scores.pivot(index='match_id', columns='innings', values='team_runs').reset_index()
final_scores['winner_innings'] = np.where(
    final_scores[2] > final_scores[1], 2,
    np.where(final_scores[2] < final_scores[1], 1, 0)
)

df = df.merge(final_scores[['match_id', 'winner_innings']], on='match_id', how='left')

df['result'] = np.where(
    (df['innings'] == 2) & (df['winner_innings'] == 2),
    1,
    np.where(
        (df['innings'] == 2) & (df['winner_innings'] == 1),
        0,
        -1   # tie
    )
)

In [1951]:
df.shape

(278203, 46)

In [1952]:
df.columns

Index(['match_id', 'season', 'start_date', 'venue', 'innings', 'ball',
       'batting_team', 'bowling_team', 'striker', 'non_striker', 'bowler',
       'runs_off_bat', 'extras', 'wides', 'noballs', 'byes', 'legbyes',
       'penalty', 'wicket_type', 'player_dismissed', 'other_wicket_type',
       'other_player_dismissed', 'curr_ball_runs', 'team_runs',
       'first_innings_runs', 'second_innings_runs', 'is_wicket', 'target',
       'over', 'ball_in_over', 'balls_bowled', 'runs_to_win', 'curr_run_rate',
       'req_run_rate', 'crr_rrr_ratio', 'balls_remaining', 'total_wickets',
       'wickets_remaining', 'is_out', 'valid_ball', 'batting_average',
       'career_strike_rate', 'exp_bowler_eco', 'exp_bowler_avg',
       'winner_innings', 'result'],
      dtype='object')

In [1953]:
drop_list = [
    'match_id', 'start_date', 'runs_off_bat', 'extras', 'wides',
    'noballs', 'byes', 'legbyes', 'penalty', 'curr_ball_runs',
    'wicket_type', 'player_dismissed', 'other_wicket_type',
    'other_player_dismissed', 'is_wicket', 'is_out', 'winner_innings',
    'ball', 'over', 'ball_in_over', 'balls_bowled', 'valid_ball',
    'team_runs', 'first_innings_runs', 'second_innings_runs', 'target',
    # 'batsman_run_cum', 'balls_faced_cum',
    'total_wickets'
]
df = df.drop(columns=drop_list)

In [1954]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 278203 entries, 0 to 278202
Data columns (total 19 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   season              278203 non-null  int64  
 1   venue               243815 non-null  object 
 2   innings             278203 non-null  int64  
 3   batting_team        278203 non-null  object 
 4   bowling_team        278203 non-null  object 
 5   striker             278203 non-null  object 
 6   non_striker         278203 non-null  object 
 7   bowler              278203 non-null  object 
 8   runs_to_win         278203 non-null  float64
 9   curr_run_rate       278203 non-null  float64
 10  req_run_rate        278203 non-null  float64
 11  crr_rrr_ratio       278203 non-null  float64
 12  balls_remaining     278203 non-null  int64  
 13  wickets_remaining   278203 non-null  float64
 14  batting_average     278203 non-null  float64
 15  career_strike_rate  278203 non-nul

In [1955]:
df['pressure'] = df['req_run_rate'] - df['curr_run_rate']

In [1956]:
df = df[df['innings'] <= 2]
df = df[df['result'] != -1]


In [1957]:
df.describe()

,season,innings,runs_to_win,curr_run_rate,req_run_rate,crr_rrr_ratio,balls_remaining,wickets_remaining,batting_average,career_strike_rate,exp_bowler_eco,exp_bowler_avg,result,pressure
count,118631.000000,118631.0,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000,118631.000000
mean,2016.780437,2.0,96.398243,7.358980,10.501400,0.916069,62.651651,7.488279,27.939213,129.659723,7.581440,26.855496,0.274768,3.142420
std,5.246009,0.0,51.407017,2.358252,5.927830,0.904235,33.508355,2.179153,8.489375,12.861483,0.537289,4.987113,0.446399,6.367773
min,2008.000000,2.0,-15.000000,0.000000,0.000000,0.000000,0.000000,1.000000,5.428571,82.530120,5.914198,14.846154,0.000000,-17.582609
25%,2012.000000,2.0,56.000000,6.200000,7.523810,0.525395,34.000000,6.000000,22.315789,121.686747,7.266833,23.195122,0.000000,-0.397487
50%,2017.000000,2.0,95.000000,7.500000,9.265823,0.759171,63.000000,8.000000,28.000000,128.448276,7.604061,26.121951,0.000000,2.225064
75%,2022.000000,2.0,134.000000,8.769231,11.550000,1.041667,92.000000,9.000000,32.907216,136.516854,7.933333,30.095238,1.000000,5.314286
max,2025.000000,2.0,286.000000,24.000000,36.000000,10.000000,119.000000,10.000000,67.333333,194.570136,9.425287,45.312500,1.000000,31.565217


In [1958]:
df = df[df['innings'] == 2]
df = df[df['runs_to_win'] >= 0]
df = df[df['balls_remaining'] > 0]

In [1959]:
df.describe()

,season,innings,runs_to_win,curr_run_rate,req_run_rate,crr_rrr_ratio,balls_remaining,wickets_remaining,batting_average,career_strike_rate,exp_bowler_eco,exp_bowler_avg,result,pressure
count,117693.000000,117693.0,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000,117693.000000
mean,2016.780633,2.0,97.074499,7.350947,10.409520,0.922278,63.111026,7.513964,27.973450,129.688260,7.580446,26.854198,0.273338,3.058573
std,5.246823,0.0,51.021485,2.362626,5.647893,0.905085,33.229394,2.159021,8.479825,12.852067,0.537559,4.987972,0.445675,6.116005
min,2008.000000,2.0,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000,5.428571,82.530120,5.914198,14.846154,0.000000,-17.582609
25%,2012.000000,2.0,57.000000,6.184615,7.531915,0.531674,35.000000,6.000000,22.363636,121.691678,7.266833,23.195122,0.000000,-0.381766
50%,2017.000000,2.0,96.000000,7.479452,9.258621,0.762821,64.000000,8.000000,28.000000,128.448276,7.602931,26.121951,0.000000,2.217391
75%,2022.000000,2.0,135.000000,8.769231,11.510204,1.044944,92.000000,9.000000,32.907216,136.597189,7.933333,30.095238,1.000000,5.266715
max,2025.000000,2.0,286.000000,24.000000,36.000000,10.000000,119.000000,10.000000,67.333333,194.570136,9.425287,45.312500,1.000000,31.565217


In [1960]:
train = df[df['season'] <= 2021]
cv    = df[(df['season'] >= 2022) & (df['season'] <= 2023)]
test  = df[(df['season'] >= 2024) & (df['season'] <= 2025)]

In [1961]:
x_train = train.drop('result', axis=1)
y_train = train['result']

x_cv = cv.drop('result', axis=1)
y_cv = cv['result']

x_test = test.drop('result', axis=1)
y_test = test['result']

In [1962]:
print("Train:", train['season'].unique())
print("CV:", cv['season'].unique())
print("Test:", test['season'].unique())

Train: [2008 2009 2010 2011 2012 2013 2014 2015 2016 2017 2018 2019 2020 2021]
CV: [2022 2023]
Test: [2024 2025]


In [1963]:
x_train = x_train.drop(columns=['season'])
x_cv = x_cv.drop(columns=['season'])
x_test = x_test.drop(columns=['season'])

In [1964]:
# from sklearn.model_selection import train_test_split
# x_train,x_temp,y_train,y_temp=train_test_split(df, y, test_size=0.4, random_state=42)
# x_cv,x_test,y_cv,y_test=train_test_split(x_temp, y_temp,test_size=0.5, random_state=42)

In [1965]:
x_train = x_train.drop(columns=['innings'])
x_cv = x_cv.drop(columns=['innings'])
x_test = x_test.drop(columns=['innings'])

In [1966]:
x_train.info()

<class 'pandas.core.frame.DataFrame'>
Index: 87152 entries, 124 to 208039
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   venue               87152 non-null  object 
 1   batting_team        87152 non-null  object 
 2   bowling_team        87152 non-null  object 
 3   striker             87152 non-null  object 
 4   non_striker         87152 non-null  object 
 5   bowler              87152 non-null  object 
 6   runs_to_win         87152 non-null  float64
 7   curr_run_rate       87152 non-null  float64
 8   req_run_rate        87152 non-null  float64
 9   crr_rrr_ratio       87152 non-null  float64
 10  balls_remaining     87152 non-null  int64  
 11  wickets_remaining   87152 non-null  float64
 12  batting_average     87152 non-null  float64
 13  career_strike_rate  87152 non-null  float64
 14  exp_bowler_eco      87152 non-null  float64
 15  exp_bowler_avg      87152 non-null  float64
 16  pressu

In [1967]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder,TargetEncoder,StandardScaler
from sklearn.linear_model import LogisticRegression
# target_encode=['striker','bowler','non_striker']
nominal_cols=['batting_team','bowling_team']#,'venue']
numeric_cols=['runs_to_win', 'curr_run_rate', 'req_run_rate', 'crr_rrr_ratio',
              'balls_remaining', 'wickets_remaining',
              #'match_strike_rate',
              'batting_average','pressure',
              'exp_bowler_eco', 'exp_bowler_avg']
print("Infinite values remaining:", np.isinf(df[numeric_cols]).values.sum())
preprocessing=ColumnTransformer(transformers=[
    ('nominal',OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'),nominal_cols),
    # ('target',TargetEncoder(cv=5),target_encode),
    ('scalar',StandardScaler(),numeric_cols)
])


Infinite values remaining: 0


In [1968]:
pipe=make_pipeline(preprocessing,LogisticRegression(C=0.001, solver='liblinear', max_iter=1000))

In [1969]:
pipe.fit(x_train,y_train)

Pipeline(steps=[('columntransformer',
                 ColumnTransformer(transformers=[('nominal',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['batting_team',
                                                   'bowling_team']),
                                                 ('scalar', StandardScaler(),
                                                  ['runs_to_win',
                                                   'curr_run_rate',
                                                   'req_run_rate',
                                                   'crr_rrr_ratio',
                                                   'balls_remaining',
                                                   'wickets_remaining',
                                                   'batting_average',
                                                   'pressure', 'exp_bowler_eco',
                                                   'exp_bowler_avg'])])),
                ('logisticregression',
                 LogisticRegression(C=0.001, max_iter=1000,
                                    solver='liblinear'))])

In [1970]:
proba_cv=pipe.predict(x_cv)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [1971]:
proba_train=pipe.predict(x_train)

In [1972]:
proba_test=pipe.predict(x_test)

/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [1973]:
from sklearn.metrics import roc_auc_score,f1_score,accuracy_score
print(accuracy_score(proba_train,y_train))
print(accuracy_score(proba_cv,y_cv))
print(accuracy_score(proba_test,y_test))

0.7390306590783918
0.8097491411162248
0.7325658330025142


In [1974]:
# from sklearn.ensemble import RandomForestClassifier,VotingClassifier
# from xgboost import XGBClassifier
# voting_clf=VotingClassifier(estimators=[
#     # ('rf',RandomForestClassifier()),
#      ('lr',LogisticRegression(max_iter=1000)),
#     ('XG',XGBClassifier())],voting='soft')

In [1975]:
# voting_pipe=make_pipeline(preprocessing,voting_clf)

In [1976]:
# from sklearn.model_selection import RandomizedSearchCV
# param_dict = {
#     # XGBoost
#     'votingclassifier__XG__n_estimators': [200, 300, 500],
#     'votingclassifier__XG__max_depth': [3, 4, 6],
#     'votingclassifier__XG__learning_rate': [0.05, 0.1, 0.2],
#     'votingclassifier__XG__subsample': [0.7, 0.8],
#     'votingclassifier__XG__reg_lambda': [1, 5, 10],
#     'votingclassifier__XG__colsample_bytree': [0.7, 0.8],

#     # # RF
#     # 'votingclassifier__rf__n_estimators': [100, 200],
#     # 'votingclassifier__rf__max_depth': [5, 6],
#     # 'votingclassifier__rf__min_samples_leaf': [20, 50],
#     # 'votingclassifier__rf__max_features': ['sqrt'],
# }
# random_search=RandomizedSearchCV(estimator=voting_pipe,param_distributions=param_dict,n_iter=20,cv=5,verbose=2,n_jobs=-1)

In [1977]:
# random_search.fit(x_train,y_train)

In [1978]:
# best_estimator=random_search.best_estimator_
# print("Best_Parameters ",random_search.best_params_)
# print("Best_Estimator",best_estimator)

In [1979]:
# test1=best_estimator.predict(x_test)
# test2=best_estimator.predict(x_train)
# test3=best_estimator.predict(x_cv)

In [1980]:
# print('best_estimator_on_cv: ',accuracy_score(test3,y_cv))
# print('best_estimator_on_train: ',accuracy_score(test2,y_train))
# print('best_estimator_on_test: ',accuracy_score(test1,y_test))

In [1981]:
# import pandas as pd

# xgb_model = best_estimator.named_steps['votingclassifier'].estimators_[2]
# features = best_estimator.named_steps['columntransformer'].get_feature_names_out()

# importance = pd.Series(xgb_model.feature_importances_, index=features).sort_values(ascending=False)
# importance.head(10).plot(kind='barh', title='What Drives a Win?')

In [1982]:
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder,TargetEncoder,StandardScaler

overfit_cols = ['striker', 'non_striker', 'bowler', 'venue']

x_ttrain = x_train.drop(columns=overfit_cols)
x_ccv    = x_cv.drop(columns=overfit_cols)
x_ttest  = x_test.drop(columns=overfit_cols)

target_encode_modified = []

nominal_cols_modified = ['batting_team','bowling_team']

preprocessing_for_xgb = ColumnTransformer(transformers=[
    ('nominal', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'), nominal_cols_modified),
    ('target', TargetEncoder(cv=5), target_encode_modified),
    ('scalar', StandardScaler(), numeric_cols)
])

xgb_pipe = make_pipeline(
    preprocessing_for_xgb,
    XGBClassifier(
         n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=5,
    reg_alpha=1,
    min_child_weight=10,
    gamma=1,
    eval_metric='logloss',
    random_state=42
    )
)

xgb_pipe.fit(x_ttrain, y_train)

print("Train:", accuracy_score(y_train, xgb_pipe.predict(x_ttrain)))
print("CV:   ", accuracy_score(y_cv,    xgb_pipe.predict(x_ccv)))
print("Test: ", accuracy_score(y_test,  xgb_pipe.predict(x_ttest)))

Train: 0.8253970075270791
CV:    0.8017112854086991
Test:  0.7127828503374355


/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/preprocessing/_encoders.py:246: UserWarning: Found unknown categories in columns [0, 1] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(


In [1983]:
x_train.columns

Index(['venue', 'batting_team', 'bowling_team', 'striker', 'non_striker',
       'bowler', 'runs_to_win', 'curr_run_rate', 'req_run_rate',
       'crr_rrr_ratio', 'balls_remaining', 'wickets_remaining',
       'batting_average', 'career_strike_rate', 'exp_bowler_eco',
       'exp_bowler_avg', 'pressure'],
      dtype='object')

In [1984]:
x_ttrain.columns

Index(['batting_team', 'bowling_team', 'runs_to_win', 'curr_run_rate',
       'req_run_rate', 'crr_rrr_ratio', 'balls_remaining', 'wickets_remaining',
       'batting_average', 'career_strike_rate', 'exp_bowler_eco',
       'exp_bowler_avg', 'pressure'],
      dtype='object')

In [1985]:
row = pd.DataFrame([{
    'batting_team': 'Kolkata Knight Riders',
    'bowling_team': 'Sunrisers Hyderabad',
    'runs_to_win': 107,
    'curr_run_rate': 11.08,
    'req_run_rate': 11.67,
    'crr_rrr_ratio': 0.95,
    'balls_remaining': 55,
    'wickets_remaining': 7,
    'batting_average': 27.0,
    'career_strike_rate': 135.0,
    'exp_bowler_eco': 7.5,
    'exp_bowler_avg': 25.0,
    'pressure': 11.67-11.08
}])


# prediction
print(xgb_pipe.predict(row))
print(xgb_pipe.predict_proba(row))

[0]
[[0.8700509  0.12994906]]


In [1986]:
row = pd.DataFrame([{
    'batting_team': 'Kolkata Knight Riders',
    'bowling_team': 'Sunrisers Hyderabad',
    'runs_to_win': 107,
    'curr_run_rate': 11.08,
    'req_run_rate': 11.67,
    'crr_rrr_ratio': 0.95,
    'balls_remaining': 55,
    'wickets_remaining': 7,
    'batting_average': 27.0,
    'career_strike_rate': 135.0,
    'exp_bowler_eco': 7.5,
    'exp_bowler_avg': 25.0,
    'pressure': 11.67-11.08
}])


# prediction
print(pipe.predict(row))
print(pipe.predict_proba(row))

[0]
[[0.78571819 0.21428181]]


In [1987]:
row = pd.DataFrame([{
    'batting_team': 'Mumbai Indians',
    'bowling_team': 'Chennai Super Kings',
    'runs_to_win': 48,
    'curr_run_rate': 8.5,
    'req_run_rate': 9.6,
    'crr_rrr_ratio': 0.88,
    'balls_remaining': 30,
    'wickets_remaining': 7,
    'batting_average': 28,
    'career_strike_rate': 140,
    'exp_bowler_eco': 7.2,
    'exp_bowler_avg': 26,
    'pressure': 9.6-8.5
}])
print(xgb_pipe.predict(row))
print(xgb_pipe.predict_proba(row))

[0]
[[0.58183205 0.41816795]]


In [1988]:
row = pd.DataFrame([{
    'batting_team': 'Mumbai Indians',
    'bowling_team': 'Chennai Super Kings',
    'runs_to_win': 48,
    'curr_run_rate': 8.5,
    'req_run_rate': 9.6,
    'crr_rrr_ratio': 0.88,
    'balls_remaining': 30,
    'wickets_remaining': 7,
    'batting_average': 28,
    'career_strike_rate': 140,
    'exp_bowler_eco': 7.2,
    'exp_bowler_avg': 26,
    'pressure': 9.6-8.5
}])
print(pipe.predict(row))
print(pipe.predict_proba(row))

[0]
[[0.60723765 0.39276235]]
